# 03 - Agent runs (Phase 3)

Runs the LangGraph agent on **one segment** of a mined run:
`load_evidence -> diagnose -> redesign -> apply_edits -> validate -> human_review -> finalize`.

- The model sees only the evidence digest (IDs + code-generated summaries), the heuristics catalogue, domain facts and constraints - never events.
- Every output is validated in code (evidence IDs, heuristic IDs, activities, protected activities, unsupported numbers) and retried with the errors fed back (max 2 retries).
- `apply_edits` and `validate` are Phase 4 stubs (`not_implemented_phase4`).
- The run pauses at `human_review`; this notebook simulates one **reject with comment** (loops back to `redesign`) and then an **approve**.

Start with the smallest segment (2-way match). Run the others only after reviewing this output.

In [ ]:
import os

import pandas as pd
from IPython.display import Markdown, display

from processpilot.agent.graph import AgentRunner
from processpilot.agent.report import (
    diagnosis_markdown,
    issue_details,
    proposals_markdown,
    validation_log,
    warnings_markdown,
)
from processpilot.agent.runlog import RunLogger
from processpilot.config import find_project_root, load_env
from processpilot.ingest.mapping import load_preset
from processpilot.mining.evidence import segment_slug

load_env()
_, preset = load_preset("bpi2019")

# Mining run with top_n_bottlenecks = 15 (re-mined before Phase 3).
RUN_ID = "20261018T212755Z"
SEGMENT = "2-way match"
run_dir = find_project_root() / "outputs" / "runs" / RUN_ID

key = os.environ.get("ANTHROPIC_API_KEY", "")
assert key and "REPLACE" not in key, "set a real ANTHROPIC_API_KEY in .env first"
print("model:", os.environ["PP_MODEL"])

## 1. Run to the review interrupt

In [ ]:
runner = AgentRunner(run_dir, preset)
print("constraints:", runner.default_constraints().model_dump())
agent_run_id, state = runner.start(SEGMENT)
log = RunLogger(run_dir / segment_slug(SEGMENT) / "agent" / agent_run_id)
print("agent run:", agent_run_id, "| status:", state.status, "| error:", state.error)
print("log:", log.directory)

## 2. Diagnosis

In [ ]:
display(Markdown(diagnosis_markdown(state.diagnosis)))

## 3. Proposals (round 1)

In [ ]:
display(Markdown(proposals_markdown(state.proposals)))
display(Markdown(warnings_markdown(state.warnings)))

## 4. Validation log (every LLM call, retries, issues)

In [ ]:
display(pd.DataFrame(validation_log(log)))
details = issue_details(log)
pd.DataFrame(details) if details else "no validation issues on any call"

## 5. Analyst rejects with a comment -> back to `redesign`

In [ ]:
COMMENT = (
    "Do not weaken financial controls or segregation of duties. Prefer changes that "
    "reduce waiting before the invoice without merging approval and invoice steps."
)
state = runner.resume(SEGMENT, agent_run_id, {"decision": "reject", "comment": COMMENT})
print("status:", state.status, "| revision:", state.revision_count, "| error:", state.error)
display(Markdown(proposals_markdown(state.proposals)))
display(Markdown(warnings_markdown(state.warnings)))

## 6. Analyst approves -> `finalize`

In [ ]:
state = runner.resume(SEGMENT, agent_run_id, {"decision": "approve", "comment": None})
print("final status:", state.status)
pd.DataFrame([r.model_dump() for r in state.review_history])

## 7. Token usage and cost (from the run log)

In [ ]:
display(pd.DataFrame(validation_log(log)))
totals = log.usage_totals()
totals

In [ ]:
sorted(p.name for p in log.directory.iterdir())

## 8. Other segments (only after reviewing the 2-way match output)

Left disabled on purpose.

In [ ]:
RUN_OTHER_SEGMENTS = False
if RUN_OTHER_SEGMENTS:
    for seg in [s for s in runner.store.segments() if s != SEGMENT]:
        rid, st = runner.start(seg)
        print(f"{seg}: {st.status} ({rid}) {st.error or ''}")

In [ ]:
runner.close()